# 第 13 章实验：去噪自编码器与表示相似度
合成条纹图，训练一个带瓶颈的小型去噪自编码器（输入 16×16 = 256 维，编码只有 `latent_dim` 维），再比较同一样本不同噪声视图的编码相似度。最后一个实验换成讲义图 1 用的条纹，对照瓶颈宽度的实测。Notebook 不给出 VAE 或 NT-Xent 作业答案。

In [ ]:
import math
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# 图里的中文需要中文字体：Windows 的字体放前面，后面是 macOS 和 Linux 的
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Heiti SC', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 有 NVIDIA 显卡就用显卡，没有就用 CPU。这个实验很小，在 CPU 上全部跑完也只要十几秒
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

torch.manual_seed(13)
samples = 64
size = 16
noise_std = 0.30
latent_dim = 8      # 瓶颈宽度：编码的维数，要远小于输入的 size*size = 256 维
train_steps = 300

In [ ]:
clean = torch.zeros(samples, 1, size, size)
for i in range(samples):
    offset = 3 + (i % 6)
    if i % 2 == 0:
        clean[i, 0, :, offset:offset+3] = 1
    else:
        clean[i, 0, offset:offset+3, :] = 1
clean = (clean + 0.03 * torch.randn_like(clean)).clamp(0, 1)
noisy = (clean + noise_std * torch.randn_like(clean)).clamp(0, 1)
clean, noisy = clean.to(device), noisy.to(device)
print('clean/noisy:', tuple(clean.shape), clean.min().item(), clean.max().item())

In [ ]:
pixels = size * size

class DenoisingAE(nn.Module):
    # 全连接的 256-128-latent_dim-128-256，和讲义图 1 用的结构相同（这里输出多一个 Sigmoid）
    def __init__(self, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(nn.Flatten(), nn.Linear(pixels, 128), nn.ReLU(), nn.Linear(128, latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 128), nn.ReLU(), nn.Linear(128, pixels), nn.Sigmoid(), nn.Unflatten(1, (1, size, size)))
    def forward(self, x):
        return self.decoder(self.encoder(x))

model = DenoisingAE(latent_dim).to(device)
print(f'输入 {pixels} 维 -> 编码 {latent_dim} 维 -> 输出 {pixels} 维；编码是输入的 1/{pixels // latent_dim}')
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
history = []
for _ in range(train_steps):
    # 去噪目标：每一步给干净图重新加一份噪声，让网络还原干净图
    batch = (clean + noise_std * torch.randn_like(clean)).clamp(0, 1)
    optimizer.zero_grad(); loss = F.mse_loss(model(batch), clean)
    loss.backward(); optimizer.step(); history.append(float(loss.detach()))
with torch.no_grad(): reconstruction = model(noisy)
print(f'带噪声的输入和干净图之间的 MSE={float(F.mse_loss(noisy, clean)):.5f}')
print(f'final reconstruction MSE={float(F.mse_loss(reconstruction, clean)):.5f}（noisy 这一份噪声训练时没有见过）')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(9, 2.5))
axes[0].imshow(clean[0,0].cpu(), cmap='gray'); axes[0].set_title('干净图')
axes[1].imshow(noisy[0,0].cpu(), cmap='gray'); axes[1].set_title('加噪声的输入')
axes[2].imshow(reconstruction[0,0].cpu(), cmap='gray'); axes[2].set_title('重建')
axes[3].plot(history); axes[3].set_title('MSE 损失')
for ax in axes[:3]: ax.axis('off')
plt.tight_layout(); plt.show()

## 自监督表示观察
把同一干净图加入两份独立噪声，比较 `latent_dim` 维编码向量的余弦相似度；再与另一类别样本比较。这里只观察，不实现对比损失。

In [ ]:
view1 = (clean[:8] + noise_std * torch.randn_like(clean[:8])).clamp(0, 1)
view2 = (clean[:8] + noise_std * torch.randn_like(clean[:8])).clamp(0, 1)
with torch.no_grad():
    z1 = model.encoder(view1)
    z2 = model.encoder(view2)
    same = F.cosine_similarity(z1, z2).mean()
    shifted = F.cosine_similarity(z1, z2.roll(1, dims=0)).mean()
print(f'same-image similarity={float(same):.3f}')
print(f'shifted-pair similarity={float(shifted):.3f}')
print('注意：重建好不保证下游表示一定好，仍需线性探测。')

## 瓶颈宽度：和讲义图 1 对照
讲义第 1 节实测过：自编码器把缺陷也重建出来，需要瓶颈宽并且训练得久；去噪目标能挡住这件事。下面用和讲义相同的设置在一个随机种子上重做一遍：良品是 16×16 的竖条纹（周期 8 px，幅度 0.3，像素噪声标准差 0.06），2000 张训练；网络是 256-128-k-128-256，Adam，学习率 0.002，批大小 256，在第 300 步和第 1500 步各评估一次。

“缺陷被重建出来的比例”：同一张图加缺陷（半径 2 px、幅度 0.12 的圆斑）前后各重建一次，看两次重建的差里有多少是缺陷本身。0 表示缺陷完全留在残差里，1 表示被原样画了出来。讲义的数字是 3 个种子的平均，这里只有一个种子，数值会有出入，先预测三行的大小关系再运行。在 CPU 上大约要十秒。

In [ ]:
generator = torch.Generator().manual_seed(13)
xx = torch.arange(size, dtype=torch.float32).expand(size, size)
yy = xx.T

def make_stripes(n):
    phase = torch.rand(n, 1, 1, generator=generator) * 2 * math.pi
    stripes = 0.3 * torch.sin(2 * math.pi * xx / 8 + phase) + 0.06 * torch.randn(n, size, size, generator=generator)
    return stripes.flatten(1)

good_train = make_stripes(2000).to(device)
good_test = make_stripes(500)
center = torch.randint(3, size - 3, (500, 2), generator=generator).float()
blob = ((yy - center[:, 0, None, None]) ** 2 + (xx - center[:, 1, None, None]) ** 2) <= 4.0
sign = torch.where(torch.rand(500, generator=generator) < 0.5, -1.0, 1.0)
defect = (blob.float() * sign[:, None, None] * 0.12).flatten(1).to(device)
good_test = good_test.to(device)
defect_test = good_test + defect

def evaluate(net):
    with torch.no_grad():
        rec_good, rec_defect = net(good_test), net(defect_test)
    rmse = float(((good_test - rec_good) ** 2).mean().sqrt())
    reproduced = float((((rec_defect - rec_good) * defect).sum(1) / (defect * defect).sum(1)).mean())
    return rmse, reproduced

def train_width(k, input_noise, checkpoints=(300, 1500)):
    torch.manual_seed(13)
    net = nn.Sequential(nn.Linear(pixels, 128), nn.ReLU(), nn.Linear(128, k), nn.Linear(k, 128), nn.ReLU(), nn.Linear(128, pixels)).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=0.002)
    results = {}
    for step in range(1, max(checkpoints) + 1):
        index = torch.randint(0, len(good_train), (256,), generator=generator).to(device)
        target = good_train[index]
        inputs = target + input_noise * torch.randn_like(target) if input_noise else target
        opt.zero_grad(); loss = ((net(inputs) - target) ** 2).mean(); loss.backward(); opt.step()
        if step in checkpoints:
            results[step] = evaluate(net)
    return results

print('像素噪声的标准差是 0.06；良品的重建误差低于它，说明网络连噪声也在重建')
for name, k, input_noise in (('瓶颈 k=2', 2, 0.0), ('瓶颈 k=128', 128, 0.0), ('瓶颈 k=128，去噪（输入加 0.3 的噪声）', 128, 0.3)):
    for step, (rmse, reproduced) in train_width(k, input_noise).items():
        print(f'{name}，训练 {step} 步：良品重建的均方根误差={rmse:.4f}，缺陷被重建出来的比例={100 * reproduced:.1f}%')

### 破坏实验建议
1. 把 `noise_std` 依次增大到 0.8、1.5、3.0，记录重建 MSE 和两个相似度。噪声大到什么程度，重建开始变差，同图和不同图的相似度开始接近？
2. 把 `latent_dim` 依次改成 1、2、32、128 重新训练，记录重建 MSE。先预测：太窄时会怎样，从 8 加宽到 128 重建还会不会变好。
3. 在“瓶颈宽度”那个单元里加一行 `('瓶颈 k=32', 32, 0.0)`，或者把去噪那一行的 0.3 改成 0.05。先预测缺陷被重建出来的比例落在哪两行之间，再和讲义图 1 的表对照。